<a href="https://colab.research.google.com/github/maria-garmonina/hpml-final-project/blob/main/Calling_get_model_rev1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install the package in editable mode
! pip install -e .

! pip install pynvml rouge_score

! pip install ibm-fms

Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1a78deabc5352a44cc9f20419f12ed7fdf7cc35e94217a32c681a7398828573d
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.2/142.2 kB 11.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
! git clone https://github.com/maria-garmonina/hpml-final-project.git

# Clone the repo manually
# ! git clone https://github.com/foundation-model-stack/foundation-model-stack.git

# Change into the directory
%cd hpml-final-project

fatal: destination path 'hpml-final-project' already exists and is not an empty directory.
/content/hpml-final-project


In [1]:
import torch
from fms.models import get_model

LOADED MY VERSION OF BAMBA


In [ ]:
# THIS DOES NOT WORK WITH OUR REPO

#  Load the Bamba-9B model from Hugging Face (assuming you’re using A100)
model = get_model(
    "hf_pretrained",
    "ibm-ai-platform/Bamba-9B",
    device_type="cuda",                  # Use "cuda" for GPU inference
    data_type=torch.bfloat16,             # Efficient format for A100 (alternative: torch.float16)

)

print("Memory allocated (GB):", torch.cuda.memory_allocated() / (1024**3))

LOADED MY VERSION OF BAMBA


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.89G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/886 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.89G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.5k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

NameError: name 'DefaultSSM' is not defined

In [2]:
# PLEASE CHECK IF THIS MAKES SENSE

from fms.modules.ssm import SSM
import fms.models.bamba as _bamba_mod
_bamba_mod.DefaultSSM = SSM # assigning this before fetching the pretrained model to not run into errors

from fms.models import get_model

model = get_model(
    "hf_pretrained",
    "ibm-ai-platform/Bamba-9B",
    #device_type="cuda",
    data_type=torch.bfloat16,
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [3]:
print(model.config)

BambaConfig(src_vocab_size=128256, emb_dim=4096, nheads=32, kvheads=8, head_dim=64, norm_eps=1e-05, nlayers=32, activation_fn='silu', attn_layer_indices=[9, 18, 27], max_expected_seq_len=262144, ntk_scaling=False, tie_heads=False, rope_theta=10000.0, p_dropout=0.0, conv_kernel=4, state_size=128, hidden_grow_factor=3.5, mamba_expand=2, mamba_n_heads=128, multiple_of=256, use_bias=False, use_conv_bias=True, n_groups=1, chunk_size=256, linear_config=None, fused_weights=True, use_chunked_ssm=False)


# code below from mentor

In [ ]:
import json
import os
import random
import requests
import torch

from fms.models import get_model
from fms.utils.tokenizers import BaseTokenizer, get_tokenizer
from fms.utils.generation import generate, pad_input_ids

project_path = "/content/drive/MyDrive/HPML/HPML Project/sharegpt_filtered.json"


# --- Helper to convert prompt to input IDs ---
def ids_for_prompt(prompt, tokenizer):
    tokens = tokenizer.tokenize(prompt)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    if tokenizer.bos_token_id != tokenizer.eos_token_id:
        ids = [tokenizer.bos_token_id] + ids
    return torch.tensor(ids, dtype=torch.long, device="cuda")

# --- Dataset download if not exists ---
def __download_file(url, filename):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Successfully downloaded {filename}")
    except requests.exceptions.RequestException as e:
        print(f"Download failed: {e}")

# --- Filter dataset to select prompts of specific length ---
def __sample_requests(prompt_list, num_requests, tokenizer, prompt_length_min, prompt_length_max, seed=None):
    # if seed is not None:
    #     random.Random(seed).shuffle(prompt_list) #dont shuffle

    filtered_dataset = []
    for i in range(len(prompt_list)):
        if len(filtered_dataset) == num_requests:
            break
        prompt = prompt_list[i]
        prompt_token_ids = ids_for_prompt(prompt, tokenizer)
        prompt_len = len(prompt_token_ids)
        if prompt_length_min <= prompt_len <= prompt_length_max:
            filtered_dataset.append((prompt, prompt_len))
    return filtered_dataset

# --- Wrapper for sampling from ShareGPT dataset ---
def sample_sharegpt_requests(dataset_path, num_requests, tokenizer, prompt_length_min=32, prompt_length_max=64, seed=None):
    if not os.path.exists(dataset_path):
        print("Downloading ShareGPT dataset...")
        __download_file(
            "https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json",
            dataset_path
        )

    with open(dataset_path, encoding='utf-8') as f:
        dataset = json.load(f)
    dataset = [d for d in dataset if len(d["conversations"]) >= 2]
    dataset = [d["conversations"][0]["value"] for d in dataset]

    with open(project_path, 'w') as file:
      json.dump(dataset, file, indent=4)


    return __sample_requests(dataset, num_requests, tokenizer, prompt_length_min, prompt_length_max, seed)

# --- Load tokenizer ---
tokenizer = get_tokenizer("ibm-ai-platform/Bamba-9B")

# --- Sample prompt(s) ---
prompts_and_sizes = sample_sharegpt_requests(
    "sharegpt.json",
    num_requests=1,
    tokenizer=tokenizer,
    prompt_length_min=1,
    prompt_length_max=1024,
    seed=0,
)

# # --- Tokenize and pad ---
# prompt_list = [ids_for_prompt(prompt, tokenizer) for prompt, _ in prompts_and_sizes]
# input_ids, padding_kwargs = pad_input_ids(prompt_list)

# # --- Load the Bamba-9B model ---
# model = get_model(
#     "hf_pretrained",
#     "ibm-ai-platform/Bamba-9B",
#     device_type="cuda",
#     data_type=torch.bfloat16,
# )

# # Optional: Compile model to improve memory usage
# model = torch.compile(model)

# # --- Generate output ---
# outputs = generate(
#     model,
#     input_ids,
#     max_new_tokens=100,
#     extra_kwargs=padding_kwargs
# )

# # --- Report peak memory used ---
# print(f"🚀 Max CUDA memory used: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Successfully downloaded sharegpt.json


In [ ]:
# import json

# def extract_first_qa(path_to_json):
#     qa_list = []
#     with open(path_to_json) as f:
#         data = json.load(f)
#     for item in data:
#         conv = item.get("conversations", [])
#         # make sure there's at least one human → one gpt
#         if len(conv) >= 2 and conv[0]["from"] == "human" and conv[1]["from"] == "gpt":
#             q = conv[0]["value"].strip()
#             a = conv[1]["value"].strip()
#             qa_list.append((f"{q}", f"{a}"))



#     return qa_list

# #usage
# qa_pairs = extract_first_qa("/content/drive/MyDrive/HPML/HPML Project/ShareGPT_V3_unfiltered_cleaned_split.json")
# # now qa_pairs is a 1D list of "prompt – answer" strings
# # e.g., ["Summarize the main ideas… – Here are the main ideas…", "How to tell if… – 1. Homogeneity…", …]
# with open("/content/drive/MyDrive/HPML/HPML Project/qa_pairs.json", "w", encoding="utf-8") as f:
#   json.dump(qa_pairs, f, indent = 2)

In [ ]:
# #Filerting
# allowed_lengths = [50,51,52]
# filtered_qa_pairs = []
# for qa_pair in qa_pairs:
#   if len(qa_pair[0]) in allowed_lengths:
#     filtered_qa_pairs.append((len(qa_pair[0]), qa_pair[0], qa_pair[1]))
# qa_pairs_test = filtered_qa_pairs

# print(filtered_qa_pairs[1])
# with open("/content/drive/MyDrive/HPML/HPML Project/qa_pairs_filtered_len.json", "w", encoding="utf-8") as f:
#   json.dump(filtered_qa_pairs, f, indent = 2)



(50, 'How do I make a software like Obsidian with Python', 'To make a software like Obsidian with Python, you will need to have a strong understanding of both Python and software development in general. Obsidian is a powerful tool for creating and managing knowledge, and making a similar software with Python would require a significant amount of work.\n\nHere are the basic steps you would need to take to make a software like Obsidian with Python:\n\n1. First, you would need to decide on the specific features and functionality that you want your software to have. Obsidian is a very feature-rich tool, so you may want to focus on creating a simpler version of the software that includes only the most essential features.\n2. Once you have a clear idea of what your software will do, you will need to start designing the user interface. This will involve creating wireframes and mockups of the various screens and features of your software, to help you visualize how it will look and work.\n3. Ne

In [ ]:
# qa_pairs[0]

[51,
 'Are you familiar with the SAS programming language?',
 'Yes, I am familiar with the SAS programming language. SAS is a statistical software suite that is widely used in data analysis, data management, and business intelligence applications. It has its own programming language, which is used to manipulate and analyze data, generate reports, and automate data-driven workflows. The language is known for its ease of use and powerful capabilities, and it is commonly used in industries such as healthcare, finance, and government.']

In [ ]:
# #load in data:
# import torch
# import pandas as pd
# import time
# import matplotlib.pyplot as plt


# MAX_NEW_TOKENS = 10 #limiting to 10 because I was scared it would combust haha

# #checks device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # #loading into datastet
# # # --- Load prompts  ---
# # df = pd.read_csv(PROMPT_CSV_PATH)
# # # Source: https://huggingface.co/datasets/fka/awesome-chatgpt-prompts
# # # Pulled in as a csv
# # # Add prompt length as a feature
# # df["prompt_length"] = df["prompt"].apply(lambda x: len(tokenizer(x)["input_ids"]))
# # results = {}

# # --- Benchmarking  ---
# print("Running benchmark...")
# for i, row in df.iterrows():
#     prompt = qa_pairs_test[i][0]

#     #Tokenize input
#     inputs = tokenizer(prompt, return_tensors="pt").to(device)
#     prompt_len = len(inputs["input_ids"][0])
#     if device.type == "cuda":
#         torch.cuda.reset_peak_memory_stats()

#     #Timing model
#     start_time = time.time()
#     with torch.no_grad():
#         output = model.generate(**inputs,max_new_tokens=prompt_len + MAX_NEW_TOKENS,do_sample=False)
#     end_time = time.time()

#     #Get latency and throughput
#     total_time = end_time - start_time
#     generated_tokens = output.shape[-1]
#     throughput = generated_tokens / total_time

#     #Bandwidth (only applicable to GPU studd)
#     if device.type == "cuda":
#         memory_used = torch.cuda.max_memory_allocated()
#         bandwidth = memory_used / total_time / 1e9  # in GB/s
#     else:
#         memory_used = 0
#         bandwidth = 0

#     generated_response = tokenizer.decode(output[0], skip_special_tokens=True)
#     #Store in dictionary for easy plotting, rounding
#     results[prompt_len] = {
#         "latency": round(total_time, 4),
#         "throughput": round(throughput, 2),
#         "bandwidth": round(bandwidth, 2),
#         "memory_used": memory_used,
#         "generated_response": generated_response

#     }

#     print(f"[{i+1}/{len(df)}] Length: {prompt_len}, Latency: {total_time:.4f}s, Throughput: {throughput:.2f} tok/s, Bandwidth: {bandwidth:.2f} GB/s")
#     print(f"Generated Response: {generated_response}")

# # --- Save results in dataframe and to csv ---
# results_df = pd.DataFrame([
#     {"prompt_length": k, **v} for k, v in results.items()
# ])

In [4]:
import json
# with open("/content/drive/MyDrive/HPML/HPML Project/qa_pairs_filtered_len.json", "r",encoding="utf-8") as f:
# changed this to fetch the data
with open("/content/drive/MyDrive/qa_pairs_filtered_len.json", "r",encoding="utf-8") as f:
  qa_pairs = json.load(f)

In [5]:
print(type(qa_pairs))

print(len(qa_pairs))

<class 'list'>
1054


In [12]:
import time
import torch
import psutil
import pynvml
import pandas as pd
from torch.profiler import profile, ProfilerActivity
from fms.utils.tokenizers import get_tokenizer
from fms.utils.generation import generate
from rouge_score import rouge_scorer

# ─── 1. Load QA pairs ───
#with open("qa_pairs.json", "r", encoding="utf-8") as f:
    #qa_pairs = json.load(f)   # each entry is [prompt_len, prompt, reference]

# ─── 2. Setup tokenizer & compile the model ───
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = get_tokenizer("ibm-ai-platform/Bamba-9B")

model.compile()

# ─── 3. NVML for GPU stats ───
pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

# ─── 4. Load metrics ───
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

# ─── 5. Helpers ───
def ids_for_prompt(prompt, tokenizer, device):
    toks = tokenizer.tokenize(prompt)
    ids  = tokenizer.convert_tokens_to_ids(toks)
    if tokenizer.bos_token_id != tokenizer.eos_token_id:
        ids = [tokenizer.bos_token_id] + ids
    return torch.tensor(ids, dtype=torch.long, device=device)

def decode_ids(ids):
    toks  = tokenizer.convert_ids_to_tokens(ids)
    return tokenizer.convert_tokens_to_string(toks)

# ─── 6. Benchmark loop ───
records = []
MAX_NEW_TOKENS = 50
#log_path = "/content/drive/MyDrive/HPML/HPML Project/original_bamba_log.txt"
# CHANGE THIS TO THE LINE ABOVE
log_path = "/content/drive/MyDrive/original_bamba_log.txt"

# Open log once (append mode)
log_file = open(log_path, "a", encoding="utf-8")

for idx, (orig_len, prompt, ref) in enumerate(qa_pairs, start=1):
    inputs = ids_for_prompt(prompt, tokenizer, device)
    input_len = inputs.size(0)

    # capture system stats
    cpu0 = psutil.cpu_percent(None)
    io0  = psutil.cpu_times_percent(None).iowait
    gpu0 = pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu
    if device.type=="cuda":
        torch.cuda.reset_peak_memory_stats()

    # profile the generate step
    with profile(
        activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_flops=True
    ) as prof:
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            out_ids = generate(
                model,
                inputs,
                max_new_tokens=input_len + MAX_NEW_TOKENS
            )
        torch.cuda.synchronize()
        t1 = time.time()

    # end system stats
    cpu1 = psutil.cpu_percent(None)
    io1  = psutil.cpu_times_percent(None).iowait
    gpu1 = pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu

    # compute metrics
    latency = t1 - t0
    gen_len = out_ids.size(0) - input_len
    throughput = gen_len / latency if latency>0 else 0.0
    peak_mem = (torch.cuda.max_memory_allocated() / 1024**2) if device.type=="cuda" else 0
    mem_bw = peak_mem / latency if latency>0 else 0.0

    # profiler FLOPs & top ops
    top_ops = prof.key_averages().table(sort_by="self_cuda_time_total", row_limit=5)
    total_flops = sum(evt.flops for evt in prof.key_averages() if hasattr(evt,"flops"))

    # decode and eval
    gen_text = decode_ids(out_ids)
    rouge_sc = scorer.score(ref, gen_text)

    # record
    rec = {
        "prompt_len": input_len,
        "gen_len": gen_len,
        "latency_s": latency,
        "throughput_tok_s": throughput,
        "peak_mem_MB": peak_mem,
        "mem_bw_MBps": mem_bw,
        "cpu_start_%": cpu0, "cpu_end_%": cpu1,
        "gpu_start_%": gpu0, "gpu_end_%": gpu1,
        "io_wait_diff_%": io1-io0,
        "total_flops": total_flops,
        "profiler_top_ops": top_ops,
        "rouge1": rouge_sc["rouge1"].fmeasure,
        "rouge2": rouge_sc["rouge2"].fmeasure,
        "rougeL": rouge_sc["rougeL"].fmeasure,
    }
    records.append(rec)

    # log one line
    log_file.write(
        f"\n==== Prompt #{idx} ====\n"
        f"Prompt: {prompt}\n"
        f"Ref   : {ref}\n"
        f"Gen   : {gen_text}\n"
        f"Metrics: {rec}\n"
    )
    print(f"{idx}/{len(qa_pairs)} | L={input_len} | lat={latency:.3f}s | ROUGE-L={rec['rougeL']:.3f}")

log_file.close()

# ─── 7. Save to CSV ───
pd.DataFrame(records).to_csv("benchmark_full_metrics.csv", index=False)

[--------------------------------------------------]


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
p = qa_pairs[0][1]
def ids_for_prompt(prompt, tokenizer):
    tokens = tokenizer.tokenize(prompt)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    if tokenizer.bos_token_id != tokenizer.eos_token_id:
        ids = [tokenizer.bos_token_id] + ids
    return torch.tensor(ids, dtype=torch.long, device="cuda")


idz = ids_for_prompt(p,tokenizer)
print(idz)
print(p)


def get_back_words(ids):
  tokens = tokenizer.convert_ids_to_tokens(ids)
  words = tokenizer.convert_tokens_to_string(tokens)
  return words


SyntaxError: expected ':' (<ipython-input-16-26d89839c9f1>, line 15)